
# AI-Based Food Delivery Time Prediction and Operational Analytics

**Academic Data Analytics with AI Internship Project**

**Dataset:** `Order_delivery.csv`

This notebook contains the complete project workflow:

**Data Loading → Data Cleaning → EDA → Feature Engineering → Machine Learning → Model Evaluation → Business Analytics → Streamlit UI**

The notebook uses the actual dataset at execution time and does not invent analytical or model results.


In [ ]:

# 1. IMPORT LIBRARIES

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")


## 2. DATA LOADING

In [ ]:

# Load the original dataset

FILE_PATH = "Order_delivery.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"{FILE_PATH} was not found. Place Order_delivery.csv in the same folder as this notebook."
    )

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())


In [ ]:

# Dataset structure

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:

# Descriptive statistics

display(df.describe(include="all").T)


## 3. DATA CLEANING

In [ ]:

# Preserve the original dataset and create a cleaning copy

df_original = df.copy()
df_clean = df.copy()

# Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

# Remove completely empty rows and columns
empty_rows = df_clean.isna().all(axis=1).sum()
empty_cols = df_clean.isna().all(axis=0).sum()

df_clean = df_clean.dropna(axis=0, how="all")
df_clean = df_clean.dropna(axis=1, how="all")

print("Completely empty rows removed:", empty_rows)
print("Completely empty columns removed:", empty_cols)


In [ ]:

# Standardize text and categorical fields

object_columns = df_clean.select_dtypes(include="object").columns.tolist()

for col in object_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()

# Convert expected numerical columns
numeric_candidates = [
    "Quantity",
    "Total_Price",
    "Delivery_Duration_Minutes",
    "Restaurant_Lat",
    "Restaurant_Lon",
    "Customer_Lat",
    "Customer_Lon",
    "Driver_Lat",
    "Driver_Lon",
    "Delivery_Distance_km"
]

for col in numeric_candidates:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Convert timestamps
for col in ["Order_Time", "Delivery_Time"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

print("Type conversion completed.")
display(df_clean.dtypes.to_frame("Data Type"))


In [ ]:

# Duplicate and missing-value checks

duplicate_count = int(df_clean.duplicated().sum())
print("Duplicate rows found:", duplicate_count)

if duplicate_count:
    df_clean = df_clean.drop_duplicates()

missing_report = df_clean.isnull().sum().sort_values(ascending=False)
print("\nMissing values after type conversion:")
display(missing_report[missing_report > 0].to_frame("Missing Values"))


In [ ]:

# Validate numerical and geographical values before removing invalid records

checks = {}

if "Quantity" in df_clean:
    checks["Quantity <= 0"] = int((df_clean["Quantity"] <= 0).sum())

if "Total_Price" in df_clean:
    checks["Total_Price < 0"] = int((df_clean["Total_Price"] < 0).sum())

if "Delivery_Duration_Minutes" in df_clean:
    checks["Delivery_Duration_Minutes <= 0"] = int(
        (df_clean["Delivery_Duration_Minutes"] <= 0).sum()
    )

if "Delivery_Distance_km" in df_clean:
    checks["Delivery_Distance_km < 0"] = int(
        (df_clean["Delivery_Distance_km"] < 0).sum()
    )

for col in ["Restaurant_Lat", "Customer_Lat", "Driver_Lat"]:
    if col in df_clean:
        checks[f"{col} outside [-90, 90]"] = int(
            ((df_clean[col] < -90) | (df_clean[col] > 90)).sum()
        )

for col in ["Restaurant_Lon", "Customer_Lon", "Driver_Lon"]:
    if col in df_clean:
        checks[f"{col} outside [-180, 180]"] = int(
            ((df_clean[col] < -180) | (df_clean[col] > 180)).sum()
        )

validation_report = pd.Series(checks, name="Invalid Count").to_frame()
display(validation_report)


In [ ]:

# Remove clearly invalid records from the modelling/analytics copy.
# The original CSV remains unchanged.

valid = pd.Series(True, index=df_clean.index)

if "Quantity" in df_clean:
    valid &= df_clean["Quantity"] > 0

if "Total_Price" in df_clean:
    valid &= df_clean["Total_Price"] >= 0

if "Delivery_Duration_Minutes" in df_clean:
    valid &= df_clean["Delivery_Duration_Minutes"] > 0

if "Delivery_Distance_km" in df_clean:
    valid &= df_clean["Delivery_Distance_km"] >= 0

for col in ["Restaurant_Lat", "Customer_Lat", "Driver_Lat"]:
    if col in df_clean:
        valid &= df_clean[col].between(-90, 90) | df_clean[col].isna()

for col in ["Restaurant_Lon", "Customer_Lon", "Driver_Lon"]:
    if col in df_clean:
        valid &= df_clean[col].between(-180, 180) | df_clean[col].isna()

rows_before = len(df_clean)
df_clean = df_clean.loc[valid].copy().reset_index(drop=True)

# Fill non-target categorical missing values with an explicit category.
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    if col != "Delivery_Duration_Minutes":
        df_clean[col] = df_clean[col].fillna("Unknown")

print("Rows removed by validity rules:", rows_before - len(df_clean))
print("Final cleaned rows:", len(df_clean))
print("Final cleaned columns:", len(df_clean.columns))


In [ ]:

# Cleaning summary

print("ORIGINAL DATASET")
print("Rows:", len(df_original))
print("Columns:", len(df_original.columns))

print("\nCLEANED DATASET")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

print("\nRemaining duplicate rows:", df_clean.duplicated().sum())

remaining_missing = df_clean.isnull().sum()
print("\nRemaining missing values:")
display(remaining_missing[remaining_missing > 0].to_frame("Missing Values"))

display(df_clean.head())


## 4. EXPLORATORY DATA ANALYSIS

In [ ]:

# Categorical distributions

categorical_columns = df_clean.select_dtypes(include=["object", "string"]).columns.tolist()

for col in categorical_columns:
    print(f"\n--- {col} ---")
    display(df_clean[col].value_counts(dropna=False).head(15).to_frame("Count"))


In [ ]:

# Delivery duration distribution

TARGET = "Delivery_Duration_Minutes"

plt.figure(figsize=(9, 5))
plt.hist(df_clean[TARGET].dropna(), bins=30)
plt.title("Distribution of Delivery Duration")
plt.xlabel("Delivery Duration (minutes)")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

display(df_clean[TARGET].describe().to_frame("Delivery Duration"))


In [ ]:

# Orders by city

if "City" in df_clean.columns:
    city_orders = df_clean["City"].value_counts()

    plt.figure(figsize=(9, 5))
    city_orders.plot(kind="bar")
    plt.title("Orders by City")
    plt.xlabel("City")
    plt.ylabel("Number of Orders")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    display(city_orders.to_frame("Orders"))


In [ ]:

# Orders by food item

if "Item_Name" in df_clean.columns:
    item_orders = df_clean["Item_Name"].value_counts().head(15)

    plt.figure(figsize=(9, 5))
    item_orders.plot(kind="bar")
    plt.title("Top Food Items by Number of Orders")
    plt.xlabel("Food Item")
    plt.ylabel("Number of Orders")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    display(item_orders.to_frame("Orders"))


In [ ]:

# Orders by traffic level

if "Traffic_Level" in df_clean.columns:
    traffic_orders = df_clean["Traffic_Level"].value_counts()

    plt.figure(figsize=(8, 5))
    traffic_orders.plot(kind="bar")
    plt.title("Orders by Traffic Level")
    plt.xlabel("Traffic Level")
    plt.ylabel("Number of Orders")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(traffic_orders.to_frame("Orders"))


In [ ]:

# Driver vehicle distribution

if "Driver_Vehicle" in df_clean.columns:
    vehicle_counts = df_clean["Driver_Vehicle"].value_counts()

    plt.figure(figsize=(8, 5))
    vehicle_counts.plot(kind="bar")
    plt.title("Driver Vehicle Distribution")
    plt.xlabel("Vehicle Type")
    plt.ylabel("Number of Drivers/Orders")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(vehicle_counts.to_frame("Count"))


In [ ]:

# Driver availability

if "Driver_Availability" in df_clean.columns:
    availability_counts = df_clean["Driver_Availability"].value_counts()

    plt.figure(figsize=(8, 5))
    availability_counts.plot(kind="bar")
    plt.title("Driver Availability")
    plt.xlabel("Availability")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(availability_counts.to_frame("Count"))


In [ ]:

# Delivery distance vs delivery duration

if {"Delivery_Distance_km", TARGET}.issubset(df_clean.columns):
    plt.figure(figsize=(9, 5))
    plt.scatter(
        df_clean["Delivery_Distance_km"],
        df_clean[TARGET],
        alpha=0.25
    )
    plt.title("Delivery Distance vs Delivery Duration")
    plt.xlabel("Delivery Distance (km)")
    plt.ylabel("Delivery Duration (minutes)")
    plt.tight_layout()
    plt.show()

    print(
        "Correlation:",
        round(df_clean["Delivery_Distance_km"].corr(df_clean[TARGET]), 4)
    )


In [ ]:

# Traffic level vs delivery duration

if {"Traffic_Level", TARGET}.issubset(df_clean.columns):
    traffic_duration = (
        df_clean.groupby("Traffic_Level")[TARGET]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    display(traffic_duration)

    plt.figure(figsize=(8, 5))
    traffic_duration["mean"].plot(kind="bar")
    plt.title("Average Delivery Duration by Traffic Level")
    plt.xlabel("Traffic Level")
    plt.ylabel("Average Delivery Duration (minutes)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


In [ ]:

# City vs average delivery duration

if {"City", TARGET}.issubset(df_clean.columns):
    city_duration = (
        df_clean.groupby("City")[TARGET]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    display(city_duration)

    plt.figure(figsize=(9, 5))
    city_duration["mean"].plot(kind="bar")
    plt.title("Average Delivery Duration by City")
    plt.xlabel("City")
    plt.ylabel("Average Delivery Duration (minutes)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:

# Vehicle type vs average delivery duration

if {"Driver_Vehicle", TARGET}.issubset(df_clean.columns):
    vehicle_duration = (
        df_clean.groupby("Driver_Vehicle")[TARGET]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    display(vehicle_duration)

    plt.figure(figsize=(8, 5))
    vehicle_duration["mean"].plot(kind="bar")
    plt.title("Average Delivery Duration by Vehicle Type")
    plt.xlabel("Vehicle Type")
    plt.ylabel("Average Delivery Duration (minutes)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 5. FEATURE ENGINEERING

In [ ]:

# Create time-based features from Order_Time

if "Order_Time" not in df_clean.columns:
    raise ValueError("Order_Time is required for feature engineering.")

df_clean["Order_Hour"] = df_clean["Order_Time"].dt.hour
df_clean["Order_DayOfWeek"] = df_clean["Order_Time"].dt.dayofweek
df_clean["Order_DayName"] = df_clean["Order_Time"].dt.day_name()
df_clean["Order_Month"] = df_clean["Order_Time"].dt.month
df_clean["Is_Weekend"] = (df_clean["Order_DayOfWeek"] >= 5).astype(int)

# Common food-delivery peak periods
peak_hours = [12, 13, 14, 19, 20, 21, 22]
df_clean["Is_Peak_Hour"] = df_clean["Order_Hour"].isin(peak_hours).astype(int)

display(
    df_clean[
        ["Order_Time", "Order_Hour", "Order_DayName", "Order_DayOfWeek",
         "Order_Month", "Is_Weekend", "Is_Peak_Hour"]
    ].head(10)
)


In [ ]:

# Peak ordering hours

hourly_orders = (
    df_clean.groupby("Order_Hour")
    .size()
    .reindex(range(24), fill_value=0)
)

plt.figure(figsize=(10, 5))
hourly_orders.plot(kind="bar")
plt.title("Orders by Hour of Day")
plt.xlabel("Order Hour")
plt.ylabel("Number of Orders")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

peak_order_hour = int(hourly_orders.idxmax())
print("Peak ordering hour:", peak_order_hour)
print("Orders during peak hour:", int(hourly_orders.max()))


## 6. MACHINE LEARNING

In [ ]:

# Define target and prediction features

TARGET = "Delivery_Duration_Minutes"

if TARGET not in df_clean.columns:
    raise ValueError(f"{TARGET} was not found.")

# Exclude post-delivery information and identifiers.
# Order_Time is represented through engineered time features.
exclude_columns = [
    TARGET,
    "Delivery_Time",
    "Order_ID",
    "User_ID",
    "Restaurant_ID",
    "Driver_ID",
    "Order_Time",
    "Order_Status"
]

feature_columns = [c for c in df_clean.columns if c not in exclude_columns]

X = df_clean[feature_columns].copy()
y = df_clean[TARGET].copy()

# Target must be available for supervised learning.
valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

print("Target:", TARGET)
print("Number of features:", len(feature_columns))
print("Features:")
print(feature_columns)
print("\nX shape:", X.shape)
print("y shape:", y.shape)


In [ ]:

# Identify feature types

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


In [ ]:

# Preprocessing pipeline

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline ready.")


### Model Training and Comparison

In [ ]:

# Train and compare three regression models

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=150,
        random_state=42
    )
}

results = []
trained_models = {}

for name, estimator in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ]
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE (minutes)": mae,
        "RMSE (minutes)": rmse,
        "R²": r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("RMSE (minutes)").reset_index(drop=True)

print("Model comparison:")
display(results_df)


In [ ]:

# Visualize model comparison

results_plot = results_df.set_index("Model")[["MAE (minutes)", "RMSE (minutes)"]]

results_plot.plot(kind="bar", figsize=(10, 5))
plt.title("Regression Model Error Comparison")
plt.xlabel("Model")
plt.ylabel("Error (minutes)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

r2_plot = results_df.set_index("Model")["R²"]
r2_plot.plot(kind="bar", figsize=(9, 5))
plt.title("R² Comparison")
plt.xlabel("Model")
plt.ylabel("R²")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:

# Select the model with the lowest RMSE

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model based on lowest RMSE:", best_model_name)

comparison = pd.DataFrame({
    "Actual Delivery Time": y_test.values,
    "Predicted Delivery Time": best_predictions
})

comparison["Absolute Error"] = (
    comparison["Actual Delivery Time"]
    - comparison["Predicted Delivery Time"]
).abs()

display(comparison.head(20))


In [ ]:

# Actual vs predicted delivery time

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_predictions, alpha=0.3)

minimum = min(y_test.min(), best_predictions.min())
maximum = max(y_test.max(), best_predictions.max())

plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.title(f"Actual vs Predicted Delivery Time - {best_model_name}")
plt.xlabel("Actual Delivery Time (minutes)")
plt.ylabel("Predicted Delivery Time (minutes)")
plt.tight_layout()
plt.show()


## 7. BUSINESS ANALYTICS

In [ ]:

# KPI calculations

kpis = {
    "Total Orders": len(df_clean)
}

if "Delivery_Duration_Minutes" in df_clean:
    kpis["Average Delivery Time (min)"] = df_clean["Delivery_Duration_Minutes"].mean()

if "Delivery_Distance_km" in df_clean:
    kpis["Average Delivery Distance (km)"] = df_clean["Delivery_Distance_km"].mean()

if "Total_Price" in df_clean:
    kpis["Average Order Value"] = df_clean["Total_Price"].mean()

if "Is_Peak_Hour" in df_clean:
    kpis["Peak-Hour Orders"] = int(df_clean["Is_Peak_Hour"].sum())

long_delivery_threshold = df_clean[TARGET].quantile(0.90)
kpis["Long Delivery Threshold (min)"] = long_delivery_threshold
kpis["Long Delivery Orders"] = int((df_clean[TARGET] >= long_delivery_threshold).sum())

kpi_df = pd.DataFrame({
    "KPI": list(kpis.keys()),
    "Value": list(kpis.values())
})

display(kpi_df)


In [ ]:

# Business summary tables: Drivers and Risk

business_tables = {}

if {"Traffic_Level", TARGET}.issubset(df_clean.columns):
    business_tables["Traffic"] = (
        df_clean.groupby("Traffic_Level")
        .agg(
            Orders=(TARGET, "size"),
            Average_Delivery_Time=(TARGET, "mean")
        )
        .sort_values("Average_Delivery_Time", ascending=False)
    )

if {"City", TARGET}.issubset(df_clean.columns):
    business_tables["City"] = (
        df_clean.groupby("City")
        .agg(
            Orders=(TARGET, "size"),
            Average_Delivery_Time=(TARGET, "mean")
        )
        .sort_values("Average_Delivery_Time", ascending=False)
    )

if {"Driver_Vehicle", TARGET}.issubset(df_clean.columns):
    business_tables["Vehicle"] = (
        df_clean.groupby("Driver_Vehicle")
        .agg(
            Orders=(TARGET, "size"),
            Average_Delivery_Time=(TARGET, "mean")
        )
        .sort_values("Average_Delivery_Time", ascending=False)
    )

for name, table in business_tables.items():
    print(f"--- {name} ---")
    display(table)


In [ ]:

# Data → Information → Insight → Decision → Action
# Generate evidence-based summaries from the actual results.

print("DATA → INFORMATION → INSIGHT → DECISION → ACTION")
print("-" * 60)

if "Traffic" in business_tables:
    traffic = business_tables["Traffic"]
    highest_traffic = traffic["Average_Delivery_Time"].idxmax()
    print(
        f"Traffic insight: '{highest_traffic}' has the highest measured "
        f"average delivery duration ({traffic.loc[highest_traffic, 'Average_Delivery_Time']:.2f} min)."
    )

if "City" in business_tables:
    city = business_tables["City"]
    highest_city = city["Average_Delivery_Time"].idxmax()
    print(
        f"City insight: '{highest_city}' has the highest measured "
        f"average delivery duration ({city.loc[highest_city, 'Average_Delivery_Time']:.2f} min)."
    )

if "Vehicle" in business_tables:
    vehicle = business_tables["Vehicle"]
    highest_vehicle = vehicle["Average_Delivery_Time"].idxmax()
    print(
        f"Vehicle insight: '{highest_vehicle}' has the highest measured "
        f"average delivery duration ({vehicle.loc[highest_vehicle, 'Average_Delivery_Time']:.2f} min)."
    )

print(
    f"Risk indicator: {kpis['Long Delivery Orders']:,} orders are at or above "
    f"the 90th-percentile delivery-duration threshold of "
    f"{kpis['Long Delivery Threshold (min)']:.2f} minutes."
)

print(
    f"Peak-period indicator: {kpis['Peak-Hour Orders']:,} orders occur during "
    f"the defined peak-hour window."
)



## 8. STREAMLIT USER INTERFACE

The following section contains the complete Streamlit application source.

The application has four pages:

1. **Overview** – KPIs and operational summary
2. **Data Analysis** – interactive charts and filters
3. **Delivery Time Prediction** – user input and prediction
4. **Business Insights** – measured findings, risks and actions

The final cell writes the UI code to `app.py`. The notebook remains the main project file.


In [ ]:

# Generate a standalone Streamlit app from this notebook.
# Run this cell after the ML section has been executed.

streamlit_code = r'''
import os
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

st.set_page_config(
    page_title="Food Delivery Analytics",
    page_icon="🍔",
    layout="wide"
)

DATA_FILE = "Order_delivery.csv"
TARGET = "Delivery_Duration_Minutes"

@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE)
    df.columns = df.columns.str.strip().str.replace(" ", "_", regex=False)

    for col in ["Quantity", "Total_Price", "Delivery_Duration_Minutes",
                "Restaurant_Lat", "Restaurant_Lon", "Customer_Lat",
                "Customer_Lon", "Driver_Lat", "Driver_Lon",
                "Delivery_Distance_km"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["Order_Time", "Delivery_Time"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string").str.strip()

    df = df.drop_duplicates().copy()

    valid = pd.Series(True, index=df.index)

    if "Quantity" in df:
        valid &= df["Quantity"] > 0
    if "Total_Price" in df:
        valid &= df["Total_Price"] >= 0
    if TARGET in df:
        valid &= df[TARGET] > 0
    if "Delivery_Distance_km" in df:
        valid &= df["Delivery_Distance_km"] >= 0

    df = df.loc[valid].copy()

    df["Order_Hour"] = df["Order_Time"].dt.hour
    df["Order_DayOfWeek"] = df["Order_Time"].dt.dayofweek
    df["Order_Month"] = df["Order_Time"].dt.month
    df["Is_Weekend"] = (df["Order_DayOfWeek"] >= 5).astype(int)
    df["Is_Peak_Hour"] = df["Order_Hour"].isin(
        [12, 13, 14, 19, 20, 21, 22]
    ).astype(int)

    return df

@st.cache_resource
def train_model(df):
    exclude = [
        TARGET, "Delivery_Time", "Order_ID", "User_ID",
        "Restaurant_ID", "Driver_ID", "Order_Time", "Order_Status"
    ]

    features = [c for c in df.columns if c not in exclude]

    X = df[features].copy()
    y = df[TARGET].copy()

    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid]

    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric),
        ("cat", categorical_pipe, categorical)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=150,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X, y)

    return model, features

df = load_data()
model, model_features = train_model(df)

st.title("🍔 AI-Based Food Delivery Time Prediction and Operational Analytics")
st.caption("Data Analytics with AI | Academic Internship Project")

page = st.sidebar.radio(
    "Select Page",
    ["Overview", "Data Analysis", "Delivery Time Prediction", "Business Insights"]
)

if page == "Overview":
    st.header("Overview")

    c1, c2, c3, c4 = st.columns(4)

    c1.metric("Total Orders", f"{len(df):,}")
    c2.metric("Avg Delivery Time", f"{df[TARGET].mean():.2f} min")
    c3.metric("Avg Distance", f"{df['Delivery_Distance_km'].mean():.2f} km")
    c4.metric("Avg Order Value", f"{df['Total_Price'].mean():.2f}")

    st.subheader("Operational Summary")

    traffic = (
        df.groupby("Traffic_Level")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )

    city = (
        df.groupby("City")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )

    col1, col2 = st.columns(2)

    with col1:
        st.write("Average Delivery Time by Traffic")
        st.bar_chart(traffic)

    with col2:
        st.write("Average Delivery Time by City")
        st.bar_chart(city)

elif page == "Data Analysis":
    st.header("Interactive Data Analysis")

    selected_city = st.selectbox(
        "City",
        ["All"] + sorted(df["City"].dropna().unique().tolist())
    )

    selected_traffic = st.selectbox(
        "Traffic Level",
        ["All"] + sorted(df["Traffic_Level"].dropna().unique().tolist())
    )

    filtered = df.copy()

    if selected_city != "All":
        filtered = filtered[filtered["City"] == selected_city]

    if selected_traffic != "All":
        filtered = filtered[filtered["Traffic_Level"] == selected_traffic]

    st.write(f"Records displayed: {len(filtered):,}")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Traffic vs Delivery Duration")
        traffic = filtered.groupby("Traffic_Level")[TARGET].mean()
        st.bar_chart(traffic)

    with col2:
        st.subheader("Vehicle vs Delivery Duration")
        vehicle = filtered.groupby("Driver_Vehicle")[TARGET].mean()
        st.bar_chart(vehicle)

    st.subheader("Delivery Distance vs Delivery Duration")
    st.scatter_chart(
        filtered[["Delivery_Distance_km", TARGET]].dropna()
        .rename(columns={TARGET: "Delivery Duration"})
        .set_index("Delivery_Distance_km")
    )

    st.subheader("Peak Ordering Hours")
    hourly = filtered.groupby("Order_Hour").size().reindex(range(24), fill_value=0)
    st.bar_chart(hourly)

elif page == "Delivery Time Prediction":
    st.header("Delivery Time Prediction")

    st.write("Enter order information available before delivery completion.")

    input_data = {}

    # Numeric inputs
    for col in ["Quantity", "Total_Price", "Delivery_Distance_km",
                "Restaurant_Lat", "Restaurant_Lon",
                "Customer_Lat", "Customer_Lon",
                "Driver_Lat", "Driver_Lon",
                "Order_Hour", "Order_DayOfWeek", "Order_Month",
                "Is_Weekend", "Is_Peak_Hour"]:
        if col in model_features:
            default = float(df[col].median()) if pd.api.types.is_numeric_dtype(df[col]) else 0.0
            if col in ["Quantity", "Order_Hour", "Order_DayOfWeek", "Order_Month",
                       "Is_Weekend", "Is_Peak_Hour"]:
                input_data[col] = st.number_input(col, value=int(round(default)))
            else:
                input_data[col] = st.number_input(col, value=float(default))

    # Categorical inputs
    for col in ["Item_Name", "City", "Payment_Method",
                "Driver_Vehicle", "Traffic_Level", "Driver_Availability"]:
        if col in model_features:
            options = sorted(df[col].dropna().astype(str).unique().tolist())
            if options:
                input_data[col] = st.selectbox(col, options)

    if st.button("Predict Delivery Time", type="primary"):
        input_df = pd.DataFrame([input_data])

        # Ensure all model features exist and preserve feature order.
        for col in model_features:
            if col not in input_df.columns:
                input_df[col] = np.nan

        input_df = input_df[model_features]

        prediction = float(model.predict(input_df)[0])

        st.success(
            f"Predicted Delivery Time: {prediction:.2f} minutes"
        )

elif page == "Business Insights":
    st.header("Business Insights")

    traffic = (
        df.groupby("Traffic_Level")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )
    city = (
        df.groupby("City")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )
    vehicle = (
        df.groupby("Driver_Vehicle")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )

    threshold = df[TARGET].quantile(0.90)
    long_orders = int((df[TARGET] >= threshold).sum())
    peak_hour = int(df.groupby("Order_Hour").size().idxmax())

    st.subheader("Data → Information → Insight → Decision → Action")

    st.markdown(
        f"- **Insight:** `{traffic.index[0]}` has the highest measured average delivery duration "
        f"among traffic levels ({traffic.iloc[0]:.2f} minutes)."
    )
    st.markdown(
        f"- **Insight:** `{city.index[0]}` has the highest measured average delivery duration "
        f"among cities ({city.iloc[0]:.2f} minutes)."
    )
    st.markdown(
        f"- **Insight:** `{vehicle.index[0]}` has the highest measured average delivery duration "
        f"among vehicle categories ({vehicle.iloc[0]:.2f} minutes)."
    )
    st.markdown(
        f"- **Risk:** {long_orders:,} orders are at or above the 90th-percentile "
        f"delivery duration ({threshold:.2f} minutes)."
    )
    st.markdown(
        f"- **Peak period:** The highest order volume occurs at hour {peak_hour}:00."
    )

    st.subheader("Recommended Operational Actions")
    st.markdown(
        "- Review driver allocation during high-order periods.\n"
        "- Monitor traffic-heavy periods and locations.\n"
        "- Pay additional attention to long-distance deliveries.\n"
        "- Use predicted delivery duration as an operational planning indicator.\n"
        "- Monitor cities and vehicle categories with higher measured delivery times."
    )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("Streamlit application written to app.py")
print("Run from the VS Code terminal with:")
print("streamlit run app.py")



## 9. PROJECT OUTPUTS AND INTERNSHIP DELIVERABLES

After executing the notebook, the main internship deliverables can include:

- `Food_Delivery_Analytics.ipynb` — complete analysis and AI/ML workflow
- `requirements.txt` — project dependencies
- `README.md` — project overview and setup instructions
- `Project_Report.docx` — professional documentation
- `app.py` — generated Streamlit user interface for demonstration

The original `Order_delivery.csv` should be kept with the project when running the notebook locally.


In [ ]:

# Final project summary

print("=" * 70)
print("AI-BASED FOOD DELIVERY TIME PREDICTION AND OPERATIONAL ANALYTICS")
print("=" * 70)
print(f"Original records : {len(df_original):,}")
print(f"Cleaned records  : {len(df_clean):,}")
print(f"Features used    : {len(feature_columns)}")
print(f"Best model       : {best_model_name}")
print(f"Best RMSE        : {results_df.iloc[0]['RMSE (minutes)']:.4f}")
print(f"Best MAE         : {results_df.iloc[0]['MAE (minutes)']:.4f}")
print(f"Best R²          : {results_df.iloc[0]['R²']:.4f}")
print("=" * 70)
